# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.


1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [4]:
from kafka import KafkaConsumer, TopicPartition
import json

# --- Configuration ---
BOOTSTRAP_SERVERS = 'localhost:29092'
TOPIC_NAME = 'json_ot'  # Using the topic from your context
PARTITION_ID = 0
OFFSET_TO_FETCH = 1184  # The specific offset you want to retrieve


In [5]:
# 1. Create a KafkaConsumer instance.
#    - `enable_auto_commit=False` is important as we are manually controlling the position.
#    - `consumer_timeout_ms` makes the consumer stop polling if no message is found after a timeout.
consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    enable_auto_commit=False,
    auto_offset_reset='earliest', # Fallback, but seek() will override it.
    consumer_timeout_ms=2000      # Stop if no message is found after 2s
)

try:
    # 2. Create a TopicPartition object for the specific partition.
    partition = TopicPartition(TOPIC_NAME, PARTITION_ID)

    # 3. Assign the consumer to that specific partition.
    #    This is required before you can seek to an offset.
    consumer.assign([partition])
    print(f"Assigned to topic '{TOPIC_NAME}' partition {PARTITION_ID}")

    # 4. Seek to the desired offset.
    #    The next message poll() fetches will be from this offset.
    consumer.seek(partition, OFFSET_TO_FETCH)
    print(f"Seeking to offset {OFFSET_TO_FETCH}...")

    # 5. Poll for the message.
    #    The consumer will fetch messages starting from the offset we specified.
    #    We can loop and break after the first one since we only need one.
    message_found = False
    for message in consumer:
        print(f"\nSuccessfully retrieved message from offset {message.offset}:")
        print(f"  Topic: {message.topic}")
        print(f"  Partition: {message.partition}")
        print(f"  Key: {message.key.decode('utf-8') if message.key else 'None'}")
        print(f"  Value: {message.value.decode('utf-8') if message.value else 'None'}")
        
        # We only wanted one message, so we break the loop.
        message_found = True
        break

    if not message_found:
        print(f"\nNo message found at offset {OFFSET_TO_FETCH} in partition {PARTITION_ID} (or consumer timed out).")

finally:
    # 6. Close the consumer to release resources.
    print("\nClosing Kafka consumer.")
    consumer.close()

Assigned to topic 'json_ot' partition 0
Seeking to offset 1184...

Successfully retrieved message from offset 1184:
  Topic: json_ot
  Partition: 0
  Key: Development
  Value: {"log": [{"t": "2025-06-27T17:29:13.957351", "level": "INFO", "message": "CREACI\u00d3N DE LA OT, se encuentra un archivo PDF de al menos tres hojas ", "detail": "Ninguno"}], "version": "0.22.0", "link": "/home/vlad/Documents/temp_borrar/aTest/Orden de trabajo Yantzaza 13-01-2025 (SB).pdf", "id_ot": 145082, "exito": true, "cuadrilla": "Yantzaza Z1 (Cuadrilla. Nro. 5)", "responsable": ["BARRAZUETA GONZAGA SERVIO GUILLERMO", "JECU"], "colaboradores": {"total": 3, "nombres": [["ALEJANDRO PACHAR AGUSTIN EDUARDO", "LIN3"], ["TORRES MALDONADO GALO DANIEL", "LIN1"], ["ZHUNAULA GUAMAN VICTOR ANIBAL", "LIN2"]]}, "diaSemana": "lunes", "fecha": "2025-01-13T00:00:00-05:00", "fechaInicio": "lunes, 13 de enero del 2025", "fechaFinal": "2025-01-13T17:10:00-05:00", "sitio": "Yantzaza-Los Encuentros", "descripcion": "Atenci\u00f3

In [ ]:

consumer = KafkaConsumer(
    'json_ot',
    bootstrap_servers=['localhost:29092'],group_id = "my-group", client_id = "kafka-consumer"
    # ... other consumer config ...
)

for message in consumer:
    raw_value = message.value
    print(f"Raw message value: {raw_value}")
    try:
        decoded_json = json.loads(raw_value.decode('utf-8'))
        print(f"Successfully decoded: {decoded_json}")
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        print(f"Problematic raw value (decoded as UTF-8 for inspection): {raw_value.decode('utf-8', errors='ignore')}")
        # You might want to log this or push to a dead-letter queue
    except UnicodeDecodeError as e:
        print(f"UnicodeDecodeError: {e}")
        print(f"Raw value causing Unicode error: {raw_value}")

In [1]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pathlib import Path
import pandas as pd
import os
import pickle


#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/find_bug'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2024'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025/restantes_febrero'
test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025/05 MAYO'


save_dir = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/db_test/'
logs_dir = save_dir + "logs/"


file_prefix = '2025-05-MAYO'


path_obj = save_dir + file_prefix + '_1_object.pkl'
path_pkl = save_dir + file_prefix + '_2_df.pkl'
path_xls = save_dir + file_prefix + '_0_df.xlsx'
path_duk = save_dir + file_prefix + '_3_df.duck'
path_pqt = save_dir + file_prefix + '_4_df.parquet'


list_pdfs = []
for path in Path( test_path ).glob("**/*.pdf"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()


Success!!!


### DASK

Primero creo un cluster locar de computación

In [2]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()

Luego, con el listado de OT's de ejecutado en primera instancia, genero objetos en los cluster de computación local, 

Ejecuto la función `load_ot()` y guardo los resultados a la misma lista de objetos

In [3]:

futures = [client.submit(gestionOT.GestionOt, file) for file in list_pdfs ]
ot_array = client.gather(futures)
ot_cargada = [ot.load_ot() for ot in ot_array]
obj_lists = client.gather(ot_cargada)



Success!!!
Success!!!
Success!!!
Success!!!


In [4]:
from eerssa import matrizActividades as mt
from eerssa import organizar as gdrive
from importlib import reload


In [5]:
# Recargar las librerias
reload(mt)
reload(gdrive)

<module 'eerssa.organizar' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/organizar.py'>

In [ ]:
ot_matrices = [ mt.ConvertirOT_a_ActividadesCSV(ot) for ot in obj_lists ]
df_total = [ df for df in ot_matrices if df is not None ]

🔥

## Renombrar PDF's y elaborar Informe de revisión

Hasta este punto tengo un listado de objetos, al procesar las actividades me genera nuevos LOGs que se guardan en el objeto y esto sirve para generar el reporte de la OT.

Es posible que necesite modificar `matrizActividades.py`

### Modelo del algoritmo.

- Necesito un directorio donde almacenar las OT con nombres y los informes.
- Verifica/Crea este directorio
- en un Loop For: 
- verifico que sea Ot valida
- obtengo los nombres de Cuadrilla y Responsable.
- comparo esos nombres con una DB externa (Google Sheets)
- Armo el nuevo nombre del archivo en un string
- Ejecuto el comando de copiar `ot:link` con el nuevo nombre.


- Tomando los LOGs de la OT
- Organizo por prioridades.
- Ejecuto `Typst` para generar el informe
- Lo guardo con el mismo nombre del archivo al final: `_REPORTE.pdf`

In [7]:
df_datos_cudarilla = gdrive.get_gsheet_df()
df_datos_cudarilla.head()

Authentication successful!
|->> Successfully opened Google Sheet: 'DB_calificar_ot'
|->> Selected worksheet by name: 'Iniciales'

|->> Data successfully imported into DataFrame


,ORDEN_RESPONSABLE,NOMBRE,INICIALES,CUADRILLA_OT,CUADRILLA_CORTO,ORDEN_CUADRILLA
0,002,MARQUEZ APOLO JHONNY FABIAN,JM,Geico,Geico,00
1,003,QUEZADA ORDOEZ ANDREW ISRAEL,AQ,IN1,IN1,00
2,004,PALACIOS MERINO ERNESTO VLADIMIR,EP,Jefatura Zonal Zamora,00_Jefe Zonal,00
4,006,RIOS RIOS FRANCISCO FERNANDO,FR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
5,007,SILVA ARMIJOS ROMEL EDUARDO,RS,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01


In [5]:
df_datos_cudarilla

,ORDEN_RESPONSABLE,NOMBRE,INICIALES,CUADRILLA_OT,CUADRILLA_CORTO,ORDEN_CUADRILLA
0,002,MARQUEZ APOLO JHONNY FABIAN,JM,Geico,Geico,00
1,003,QUEZADA ORDOEZ ANDREW ISRAEL,AQ,IN1,IN1,00
2,004,PALACIOS MERINO ERNESTO VLADIMIR,EP,Jefatura Zonal Zamora,00_Jefe Zonal,00
4,006,RIOS RIOS FRANCISCO FERNANDO,FR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
5,007,SILVA ARMIJOS ROMEL EDUARDO,RS,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
6,008,LEON CUEVA LUIS ALBERTO,LL,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
7,009,LIMA AVILA EDWIN STALIN,EL,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
8,010,RIVERA GUAMAN SEGUNDO PATRICIO,SR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
10,012,MORALES RIVERA LUIS ALBERTO,LM,Zamora Z1 (Cuadrilla. AP Nro. 4),Zamora Alumbrado,02
11,013,YAURE DIAZ RAMIRO GABRIEL,RY,Zamora Z1 (Cuadrilla. AP Nro. 4),Zamora Alumbrado,02


In [116]:
# Get One PDF to Object

one = gestionOT.GestionOt( list_pdfs[15] )
one = one.load_ot()
gdrive.renombrar_ot( one, df_datos_cudarilla )

'2_Arjan_software_design_guide.pdf'

In [111]:
one.link

'/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/2_Arjan_software_design_guide.pdf'

In [ ]:
gdrive.get_nombre_corto_cuadrilla("Líneas Energizadas (Cuadrilla  Nro.6)", df_datos_cudarilla)

'Líneas Energizadas (Cuadrilla  Nro.6)'

In [96]:
os.path.basename(one.link)

'2_Arjan_software_design_guide.pdf'

In [1]:
reload(gdrive)

NameError: name 'reload' is not defined

In [71]:
gdrive.renombrar_ot( one, df_datos_cudarilla )

'OT [03] Cuadrilla Yacuambi 2022-06-16 (017) NL.pdf'

### Definición de funciones para obtener los datos desde Google_Sheet df

In [6]:
ot_2024 = "/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/db_test/2024_v0.12_0_df.pkl"

df = pd.read_pickle( ot_2024 )

# Cambia el nombre a la Cuadrila AP y luego achica los nombres de las Cuadrillas
df['Cuadrilla'] = df['Cuadrilla'].apply( lambda x: x.replace("Zamora Z1 (Cuadrilla. AP", "Zamora Z1 AP (Cuadrilla AP") if isinstance(x, str) else x )
df['Cuadrilla'] = df['Cuadrilla'].apply( lambda s: s.split('(')[0] )

df['Cuenta']         = pd.Categorical(df.Cuenta)
df['Dia']            = pd.Categorical(df.Dia)
df['Alimentador']    = pd.Categorical(df.Alimentador)
df['Tipo']           = pd.Categorical(df.Tipo)
df['Actividad']      = pd.Categorical(df.Actividad)
df['Cuadrilla']      = pd.Categorical(df.Cuadrilla)
df['Responsable']    = pd.Categorical(df.Responsable)
df['Vehiculo']       = pd.Categorical(df.Vehiculo)
df['anio_mes']       = df['Fecha'].apply( lambda x: x[:7] if len(x) > 7 else x)
df['anio_mes']       = pd.Categorical(df.anio_mes)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41859 entries, 0 to 41858
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Item           41859 non-null  int64   
 1   Cuenta         41859 non-null  category
 2   Evento         41859 non-null  object  
 3   Materiales     41859 non-null  object  
 4   Alimentador    41859 non-null  category
 5   Primario       41859 non-null  object  
 6   Desconexion    41859 non-null  object  
 7   SIG            41859 non-null  object  
 8   Tipo           41859 non-null  category
 9   Actividad      41859 non-null  category
 10  Cuadrilla      41859 non-null  category
 11  Dia            41859 non-null  category
 12  Fecha          41859 non-null  object  
 13  InicioEvento   41859 non-null  object  
 14  FinEvento      41859 non-null  object  
 15  Responsable    41859 non-null  category
 16  Colaboradores  41859 non-null  int64   
 17  HorasExtra     41859 non-null  

In [ ]:
# Get me all the unique values for column 'Responsable'

unique_Responsable = df["Responsable"].unique()
for name in unique_Responsable:
  print(f"{name}")

In [ ]:
unique_Responsable = df["Responsable"].unique()
for name in unique_Responsable:
  print(f"{name}")

🔥
> TODO
> Imprimir un reporte de las OT que no fue exitoso su conversion a OT. informar las novedades encontradas

## FIN Renombrar PDF's EOL

In [8]:
# Combined full df for exporting to excel

combined_df = pd.concat(df_total, ignore_index=True)

In [9]:
# Guardao los resultados a disco
with open( path_obj, 'wb' ) as fp:
  pickle.dump( obj_lists, fp )

combined_df.to_pickle( path_pkl )

### Debug horas extra

In [22]:
finde = combined_df[combined_df["id_ot"] == 148704]
mad = combined_df[combined_df["id_ot"] == 149102]
mad2 = combined_df[combined_df["id_ot"] == 150050]


In [26]:
mad2[["Evento","InicioEvento","FinEvento","HorasExtra"]]

,Evento,InicioEvento,FinEvento,HorasExtra
1676,Se coordina los trabajos con el ing. Ernesto P...,2025-03-26 05:00:00,2025-03-26 08:00:00,Si
1677,"Se coordina con Carlos Quiroga, para arreglar ...",2025-03-26 08:00:00,2025-03-26 14:30:00,No
1678,LUNCH en Héroes Del Cóndor.,2025-03-26 14:30:00,2025-03-26 15:30:00,No
1679,DAÑO REPORTADO POR EL Centro de Control. Mensa...,2025-03-26 15:30:00,2025-03-26 17:30:00,No
1680,"Guayzimi, se arregla llanta ponchada, en la es...",2025-03-26 17:30:00,2025-03-26 18:25:00,Si
1681,DAÑO REPORTADO POR EL Centro de Control. Mensa...,2025-03-26 18:25:00,2025-03-26 19:35:00,Si
1682,"Zumbi, se cambia tirafusible de 12A tipo T en ...",2025-03-26 19:35:00,2025-03-26 19:50:00,Si
1683,Se retorna a Yantzaza,2025-03-26 19:50:00,2025-03-26 20:10:00,Si
1684,"SE LABORA: AO, JA, JLM de 05:00 a 14:30 y de 1...",2025-03-26 00:00:01,2025-03-26 00:00:02,Si


### Fin debug horas Extra

## EXCELL Export

Primero importar el dataframe

Segundo Pintar el dataframe

Agrupar por mes

Exportar

POR HACER



In [10]:
df = pd.read_pickle( path_pkl )

# Cambia el nombre a la Cuadrila AP y luego achica los nombres de las Cuadrillas
df['Cuadrilla'] = df['Cuadrilla'].apply( lambda x: x.replace("Zamora Z1 (Cuadrilla. AP", "Zamora Z1 AP (Cuadrilla AP") if isinstance(x, str) else x )
df['Cuadrilla'] = df['Cuadrilla'].apply( lambda s: s.split('(')[0] )

df['Cuenta']         = pd.Categorical(df.Cuenta)
df['Dia']            = pd.Categorical(df.Dia)
df['Alimentador']    = pd.Categorical(df.Alimentador)
df['Tipo']           = pd.Categorical(df.Tipo)
df['Actividad']      = pd.Categorical(df.Actividad)
df['Cuadrilla']      = pd.Categorical(df.Cuadrilla)
df['Responsable']    = pd.Categorical(df.Responsable)
df['Vehiculo']       = pd.Categorical(df.Vehiculo)
df['anio_mes']       = df['Fecha'].apply( lambda x: x[:7] if len(x) > 7 else x)
df['anio_mes']       = pd.Categorical(df.anio_mes)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3744 entries, 0 to 3743
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Item           3744 non-null   int64   
 1   Cuenta         3744 non-null   category
 2   Evento         3744 non-null   object  
 3   Actividad      3744 non-null   category
 4   Alimentador    3744 non-null   category
 5   Primario       3744 non-null   object  
 6   Desconexion    3744 non-null   object  
 7   SIG            3744 non-null   object  
 8   Tipo           3744 non-null   category
 9   Materiales     3744 non-null   object  
 10  Cuadrilla      3744 non-null   category
 11  Dia            3744 non-null   category
 12  Fecha          3744 non-null   object  
 13  InicioEvento   3744 non-null   object  
 14  FinEvento      3744 non-null   object  
 15  Duracion       3744 non-null   int64   
 16  Responsable    3744 non-null   category
 17  Colaboradores  3744 non-null   in

In [11]:
df['anio_mes'].unique()

['2025-05']
Categories (1, object): ['2025-05']

In [12]:
rows_with_dot = df[df['anio_mes'] == '·']
rows_with_dot

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,uuid,anio_mes


In [13]:
df.sample(1)['InicioEvento']

948    2025-05-27 11:50:00
Name: InicioEvento, dtype: object

In [14]:
#
#  EXPORTAR A EXCEL
#

import xlsxwriter

# Create an ExcelWriter object
writer = pd.ExcelWriter( path_xls, engine='xlsxwriter')  


# Group the DataFrame by the categorical column
grouped = df.groupby('Cuadrilla', observed=False)

# Iterate through groups and write to separate sheets
for category, group_data in grouped:
    group_data.to_excel(writer, sheet_name=str(category), index=False)  

# Save the Excel file
writer.close()


# TESTING

In [ ]:
df[['Item', 'InicioEvento', 'FinEvento' ]]

,Item,InicioEvento,FinEvento
0,1,2025-02-22 00:00:01,2025-02-22 00:00:02
1,2,2025-02-22 12:18:00,2025-02-22 14:29:00
2,5,2025-02-22 00:00:01,2025-02-22 00:00:02
3,1,2025-02-24 14:00:00,2025-02-24 15:00:00
4,2,2025-02-24 15:00:00,2025-02-24 17:00:00
...,...,...,...
933,2,2025-02-28 10:36:00,2025-02-28 13:01:00
934,4,2025-02-28 13:01:00,2025-02-28 13:56:00
935,5,2025-02-28 13:56:00,2025-02-28 15:25:00
936,7,2025-02-28 15:25:00,2025-02-28 17:05:00


In [14]:
df.query("FinEvento == '·'")

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,uuid,anio_mes
539,10,MEDIDORES,Aprovechando que se esta en el sitio se instal...,PROG,Paquisha,No,No,No,EXPANSION,·,...,-1,OCHOA JARAMILLO ANGEL CLAUDIO,2,No,4-62,Yantzaza - Paquisha - Bellavista.,148321,Orden de trabajo Paquisha 28-01-2025 (AO).pdf,5e089466-ab6a-49c1-9f48-095574963046,2025-02-28T00:00:00-05


In [ ]:
[print( file ) for file in list_pdfs]

In [ ]:
[print( f" Ot Link: {ot.log} ") for ot in obj_lists]

### Generar la Matriz de Actividades para un objeto

In [8]:
nro_ot = 0
obj_lists[nro_ot].data["log"]

[{'t': '2025-03-25T00:37:38.203106',
  'level': 'INFO',
  'message': 'Se encuentra un archivo PDF de al menos tres hojas ',
  'detail': 'Ninguno'},
 {'t': '2025-03-25T00:37:38.214969',
  'level': 'ERROR',
  'message': 'No coinciden la Fecha de la OT viernes, 11 de febrero del 2022 con Fecha de Inicio: sábado, 12 de febrero del 2022',
  'detail': '|>> Comparando las dos fechas de Hoja Uno <<|'}]

In [7]:
test = obj_lists[nro_ot].load_ot()
actividades = pd.DataFrame(test.data["actividades"])

In [7]:
fechaModa = test.data['fecha']
type(fechaModa)

str

In [18]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
#matriz_test[['Cuenta','Evento','Fecha','InicioEvento','FinEvento']]
#matriz_test[['Fecha','InicioEvento','corregir_fechaInicio','FinEvento','corregir_fechaFin']]
matriz_test[['Fecha','InicioEvento','FinEvento']]

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Fecha,InicioEvento,FinEvento
0,2024-05-22 00:00:00,2024-05-22 08:00:00,2024-05-22 08:22:00
2,2024-05-22 00:00:00,2024-05-23 08:22:00,2024-05-23 08:33:00
3,2024-05-22 00:00:00,2024-05-22 08:53:00,2024-05-22 09:28:00
5,2024-05-22 00:00:00,2024-05-22 09:28:00,2024-05-22 09:59:00
7,2024-05-22 00:00:00,2024-05-22 09:59:00,2024-05-22 10:08:00
8,2024-05-22 00:00:00,2024-05-22 10:08:00,2024-05-22 10:39:00
9,2024-05-22 00:00:00,2024-05-22 10:39:00,2024-05-22 11:35:00
10,2024-05-22 00:00:00,2024-05-22 11:35:00,2024-05-22 12:30:00
11,2024-05-22 00:00:00,2024-05-22 12:30:00,2024-05-22 13:30:00
12,2024-05-22 00:00:00,2024-05-22 13:30:00,2024-05-22 14:18:00


In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar

In [16]:
test.data['fechaFinal']

'11/02/2022 23:00:00'

In [26]:
from datetime import datetime
from pytz import timezone

fechaFinal = test.data['fechaFinal']

datetime_object = datetime.strptime(fechaFinal, '%d/%m/%Y %H:%M:%S')
ecuador = timezone("America/Guayaquil")
local_datetime = ecuador.localize(datetime_object)
fechaFinal = local_datetime.isoformat()
solofechaFinal = fechaFinal.split('T')[1]
solofechaFinal

'23:00:00-05:00'

### Secuencial GLOBAL LOCK

In [ ]:
### Secuencial en un solo procesador.
obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )

## TEST TEST

In [2]:
date = "2025-03-05 09:00:00"